In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyodbc
import openpyxl

In [2]:
fpd = pd.read_csv('/Users/starsrain/2025_concord/yieldCurve_augmenting/projection_data/fpd_1106.csv')
data = pd.read_csv('/Users/starsrain/2025_concord/yieldCurve_augmenting/raw_data/0213_all_appDate.csv')

In [3]:
data.columns

Index(['Application_ID', 'PortFolioID', 'LoanID', 'Frequency', 'LPCampaign',
       'OriginatedAmount', 'AppYear', 'AppMonth', 'AppWeek', 'ApplicationDate',
       'FPDFA', 'FPDAA', 'TotalRealizedPayin', 'InstallmentNumber',
       'PaidOffPaymentAmount', 'DueDate', 'TransactionDate', 'PmtYear',
       'PmtMonth', 'Days_Since_Orig', 'Weeks_Since_Orig', 'PaymentType',
       'Payment_Number', 'PaymentStatus', 'weeks_between_orig_now',
       'CustType'],
      dtype='object')

## 90-day payin target (per payin_projection_project_handoff.md)

- **Target**: `payin_90d_ratio = cumulative_paid_by_day90 / OriginatedAmount`
- **Rule**: Sum `PaidOffPaymentAmount` for each loan over rows where `Days_Since_Orig <= 90`; divide by loan's `OriginatedAmount`.
- **Eligibility**: Only loans with at least 90 days of observation (so we know the full 90-day outcome).

In [6]:
# --- 90-day payin target preparation ---
# Optional: count only posted payments (PaymentStatus == 'D'). Set to False to sum all PaidOffPaymentAmount.
ONLY_POSTED_PAYMENTS = True
DAYS_CUTOFF = 90
# Optional cap on ratio (e.g. 1.7); set None to leave uncapped.
PAYIN_RATIO_CAP = None

# Drop rows missing required fields
df = data.dropna(subset=['LoanID', 'OriginatedAmount', 'PaidOffPaymentAmount', 'Days_Since_Orig'])
df['Days_Since_Orig'] = df['Days_Since_Orig'].astype(int)
if ONLY_POSTED_PAYMENTS and 'PaymentStatus' in df.columns:
    df = df[df['PaymentStatus'] == 'D']

# Cumulative paid by day 90 per loan
payin_90d = (
    df.loc[df['Days_Since_Orig'] <= DAYS_CUTOFF]
    .groupby('LoanID', as_index=False)
    .agg(cumulative_paid_by_day90=('PaidOffPaymentAmount', 'sum'))
)
# Loan-level attributes (first row per loan)
loan_attrs = df.groupby('LoanID', as_index=False).agg(
    OriginatedAmount=('OriginatedAmount', 'first'),
    Application_ID=('Application_ID', 'first'),
    Frequency=('Frequency', 'first'),
    CustType=('CustType', 'first'),
    AppYear=('AppYear', 'first'),
    AppMonth=('AppMonth', 'first'),
    AppWeek=('AppWeek', 'first'),
    ApplicationDate=('ApplicationDate', 'first'),
    FPDFA=('FPDFA', 'first'),
    FPDAA=('FPDAA', 'first'),
    max_Days_Since_Orig=('Days_Since_Orig', 'max'),
)
# Merge and compute ratio
train_raw = loan_attrs.merge(payin_90d, on='LoanID', how='left')
train_raw['cumulative_paid_by_day90'] = train_raw['cumulative_paid_by_day90'].fillna(0.0)
train_raw['payin_90d_ratio'] = train_raw['cumulative_paid_by_day90'] / train_raw['OriginatedAmount']
if PAYIN_RATIO_CAP is not None:
    train_raw['payin_90d_ratio'] = train_raw['payin_90d_ratio'].clip(upper=PAYIN_RATIO_CAP)

# Restrict to loans with at least 90 days of observation (known outcome)
train_90d = train_raw[train_raw['max_Days_Since_Orig'] >= DAYS_CUTOFF].copy()
train_90d = train_90d.drop(columns=['max_Days_Since_Orig'])
print('Loan-level rows (eligible for 90d target):', len(train_90d))
print('payin_90d_ratio: min={:.4f}, max={:.4f}, mean={:.4f}'.format(
    train_90d['payin_90d_ratio'].min(), train_90d['payin_90d_ratio'].max(), train_90d['payin_90d_ratio'].mean()))
train_90d.head()

/var/folders/b7/d1qg7gm9009f_cps7l193hch0000gn/T/ipykernel_65286/4245952251.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Days_Since_Orig'] = df['Days_Since_Orig'].astype(int)


Loan-level rows (eligible for 90d target): 55097
payin_90d_ratio: min=0.0000, max=7.6000, mean=1.6433


,LoanID,OriginatedAmount,Application_ID,Frequency,CustType,AppYear,AppMonth,AppWeek,ApplicationDate,FPDFA,FPDAA,cumulative_paid_by_day90,payin_90d_ratio
4,I1529365-0,1500.0,85580965,B,NEW,2023,1,1,2023-01-01 10:48:01.000,0.0,0.0,1812.5,1.208333
6,I1529367-0,1500.0,71206276,B,REPEAT,2023,1,1,2023-01-01 11:20:13.000,0.0,0.0,3550.0,2.366667
7,I1529369-0,1000.0,71206278,ME,NEW,2023,1,1,2023-01-01 12:53:49.000,NaN,0.0,500.0,0.500000
8,I1529371-0,800.0,71206284,B,NEW,2023,1,1,2023-01-01 14:15:17.000,0.0,0.0,1405.0,1.756250
13,I1529379-0,800.0,104209643,W,REPEAT,2023,1,1,2023-01-01 22:05:52.000,0.0,0.0,1953.0,2.441250


In [7]:
# Optional binary target for threshold-based use (handoff)
train_90d['payin_90d_fully_paid'] = (train_90d['payin_90d_ratio'] >= 1.0).astype(int)
# Sanity: mean payin_90d_ratio by segment (aligns with handoff cohort dimensions)
train_90d.groupby(['CustType', 'Frequency'], dropna=False)['payin_90d_ratio'].agg(['mean', 'count'])

mean  count
CustType Frequency                 
NEW      B          1.698275  20725
         MB         1.128028   2545
         ME         1.142030   1553
         W          2.061757   3678
         NaN        1.600000      1
RENEWAL  B          1.786644     17
         MB         1.291644      6
         ME         0.693510      4
REPEAT   B          1.691499  17640
         MB         1.037568   3414
         ME         1.036144   2061
         W          2.188339   3453